In [44]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from joblib import dump
import numpy as np
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.float_format", lambda x: "%0.3f" % x)
np.set_printoptions(suppress=True)

def print_line(): print("-" * 15)

In [3]:
data = pd.read_csv("Data/processed_data")
data

,home_ownership,purpose,annual_income,int_rate,loan_amount,is_loss,DTI,annual_income_ru,loan_ammount_ru,int_rate_ru
0,RENT,car,30000.000,0.153,2500,1,0.083,444000.000,37000.000,0.219
1,RENT,car,48000.000,0.186,3000,0,0.062,710400.000,44400.000,0.267
2,RENT,car,50000.000,0.160,12000,1,0.240,740000.000,177600.000,0.229
3,MORTGAGE,car,42000.000,0.106,4500,0,0.107,621600.000,66600.000,0.153
4,MORTGAGE,car,83000.000,0.060,3500,0,0.042,1228400.000,51800.000,0.086
...,...,...,...,...,...,...,...,...,...,...
38568,MORTGAGE,other,100000.000,0.130,24250,0,0.242,1480000.000,358900.000,0.186
38569,RENT,other,50000.000,0.135,25200,0,0.504,740000.000,372960.000,0.193
38570,RENT,other,65000.000,0.175,25000,0,0.385,962000.000,370000.000,0.251
38571,RENT,other,368000.000,0.182,24000,0,0.065,5446400.000,355200.000,0.262


In [37]:
X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "home_ownership", "purpose", "DTI"]] 
Y = data["is_loss"]

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.15,stratify=Y, random_state=42)
print(f"X_train: {X_train.shape}\nX_test: {X_test.shape}\nY_train: {Y_train.shape}\nY_test: {Y_test.shape}")
print(X_train.head())

X_train: (32787, 6)
X_test: (5786, 6)
Y_train: (32787,)
Y_test: (5786,)
       annual_income_ru  loan_ammount_ru  ...             purpose   DTI
13836       1184000.000       229400.000  ...  Debt consolidation 0.194
22927        666000.000       296000.000  ...  Debt consolidation 0.444
25687        740000.000       222000.000  ...               other 0.300
3452         828800.000        71040.000  ...         credit card 0.086
3508         444000.000       148000.000  ...         credit card 0.333

[5 rows x 6 columns]


Преобразуем числовые данные в единый масштаб (Нормализация). Долго (Заменено готовой библиотекой)

In [6]:
# X_train_norm = X_train.copy()
# X_test_norm = X_test.copy()
# for col in X_train.columns[:3]:
#     X_train_norm[col] = X_train_norm[col].apply(lambda val: (val - min) / (max - min))[col]
#     X_test_norm[col] = X_test_norm[col].apply(lambda val: (val - min) / (max - min))[col]
# X_train_norm.to_csv("Data/X_train_norm", index= False)
# X_test_norm.to_csv("Data/X_test_norm", index= False)

In [7]:
# X_train_norm = pd.read_csv("Data/X_train_norm")
# X_test_norm = pd.read_csv("Data/X_test_norm")

In [47]:
# model = RandomForestClassifier(n_estimators= 100, max_depth= 4,
#                                random_state=42,
#                                class_weight=dict(enumerate(class_weights)))
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, [0,1,2, 5]),
        ('cat', categorical_transformer, [3,4])
    ]
)

# class_weights = compute_class_weight(class_weight="balanced", classes = np.unique(Y_train), y=Y_train)

"""dict(enumerate(class_weights))"""

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=500, 
                                          class_weight = "balanced_subsample",
                                          criterion="entropy",
                                          ccp_alpha=0.02,
                                          max_depth=10))
])

In [31]:
param_dist = {
    "classifier__min_samples_split": range(5,30),
    "classifier__min_samples_leaf": range(5, 15),
    "classifier__max_features":[0.3,0.4,0.5] 
}
random_search = RandomizedSearchCV(estimator=pipeline, 
                            param_distributions= param_dist,
                             n_iter= 100,
                             scoring="f1",
                             cv=StratifiedKFold(random_state=42, shuffle=True),
                             random_state=42,
                             n_jobs=-1,
                             verbose=1)

random_search.fit(X_train, Y_train)
print(random_search.best_params_)
#Итог
# classifier__min_samples_split = 11
# classifier__min_samples_leaf = 17
# classifier__max_depth = 15

Fitting 5 folds for each of 100 candidates, totalling 500 fits


c:\Github\Credit-s-RIsk-Prediction\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


{'classifier__min_samples_split': 27, 'classifier__min_samples_leaf': 8, 'classifier__max_features': 0.3}


In [76]:
# min_samples_split = 10
# min_samples_leaf = 10
# max_depth = 10
# estimators = 500
# max_features = 4
min_samples_split = 30
min_samples_leaf = 20
max_depth = 7
estimators = 500
max_features = 0.5
best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=estimators, class_weight="balanced_subsample",
                                          min_samples_split=min_samples_split,
                                          min_samples_leaf=min_samples_leaf,
                                          max_depth=max_depth,
                                          max_features=max_features,
                                          criterion="entropy"))
])

In [77]:
metrics = ["precision", "recall", "roc_auc", "f1"]
cross_val_result = cross_validate(best_pipeline,
                                    X_train, Y_train,
                                   cv=StratifiedKFold(n_splits=10,shuffle=True, random_state=42),
                                   scoring=metrics,
                                   return_train_score=True,
                                   n_jobs=-1)

Лишний Раз не запускать!

In [8]:
checking_metrics = ['test_roc_auc', 'train_roc_auc', "test_f1", "train_f1", "train_precision", "test_precision",
               "train_recall", "test_recall"]
def print_dict(results: dict):
    for key, value in results.items():
        print(key + ": " + f"{value}")

def get_crossval_results()->dict:
    result = {}
    for metric in checking_metrics:
        result[metric] = float(cross_val_result.get(metric).mean())
    return result
metrics_results = []

In [78]:
if len(metrics_results) == 0:
    metrics_results = [get_crossval_results()]
    print_dict(metrics_results[0])
elif len(metrics_results) == 1:
    metrics_results.append(get_crossval_results())
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")
    
elif len(metrics_results) == 2:
    metrics_results[0] = metrics_results[1]
    metrics_results[1] = get_crossval_results()
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")

test_roc_auc: 0.6796893262977752⬆️
train_roc_auc: 0.7169245423533714⬇️
test_f1: 0.31501762343965634⬇️
train_f1: 0.3369233044482096⬇️
train_precision: 0.2236686574040938⬇️
test_precision: 0.20916886930736805⬇️
train_recall: 0.6825746350268472⬇️
test_recall: 0.6379934066575255⬆️


In [74]:
best_pipeline.fit(X_train, Y_train)
prediction = best_pipeline.predict(X_test)
f1 = f1_score(Y_test, prediction)
recall = recall_score(Y_test, prediction)
precision = precision_score(Y_test, prediction)
print(f"f1: {f1}\nRecall: {recall}\nPrecision: {precision}")

f1: 0.3243616287094548
Recall: 0.5875
Precision: 0.22402287893231648


Подбираем лучшие параметры для модели (ОЧЕНЬ ДОЛГО). Результат: n_estimators = 106, max_depth = 4.Заменено библиотекой

In [ ]:
# best_f = 0
# for estimators in range(10, 150, 3):
#     for depth in range(3, 15):
#         test_model = RandomForestClassifier(n_estimators=estimators, max_depth= depth, 
#                                             random_state=42,
#                                             class_weight=dict(enumerate(class_weights)))
#         test_model.fit(X_train_norm, Y_train)
#         f_score = fbeta_score(Y_test, test_model.predict(X_test_norm), beta=1.5)
#         if f_score > best_f:
#             best_f = f_score
#             best_estimators = estimators
#             best_max_depth = depth

# print(f"Лучший параметр estimators: {best_estimators}\nЛучший параметр max_depth: {best_max_depth}")

In [31]:
# prediction = model.predict(X_test)
# prediction = pd.DataFrame(prediction, columns=['Prediction'])
# results = pd.concat([X_test, prediction["Prediction"], Y_test], axis= 1, join="inner")
# results.rename(columns={"is_loss": "Real Value"}, inplace= True) 
# results[(results["Real Value"] == 1) & (results["Prediction"] == 1)]

In [75]:
model_path = "models/RandomForest.joblib"
dump(best_pipeline, model_path)

['models/RandomForest.joblib']